---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！

# 02 - 自定义 Dataset

## 学习目标

1. 掌握图像数据集的组织方式
2. 学会创建自定义图像 Dataset
3. 理解 transform 的作用和使用
4. 掌握三种常见的数据集组织形式

## 1. 数据集的三种常见组织形式

### 形式 1：TXT 文件

```
train.txt 内容：
data/images/cat_001.jpg 0
data/images/dog_001.jpg 1
data/images/cat_002.jpg 0
```

**优点**：简单直观
**缺点**：需要手动维护

### 形式 2：文件夹结构（最常用）

```
train/
├── cat/
│   ├── img1.jpg
│   └── img2.jpg
└── dog/
    ├── img3.jpg
    └── img4.jpg
```

**优点**：直观，易管理
**缺点**：不适合多标签任务

### 形式 3：CSV 文件（最灵活）

```
data.csv 内容：
img_name,label,split
img1.jpg,0,train
img2.jpg,1,valid
```

**优点**：灵活，支持多种元信息
**缺点**：需要 pandas

## 2. 形式 1：TXT 文件格式的 Dataset

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

class TXTDataset(Dataset):
    """从 TXT 文件读取图像路径和标签"""
    
    def __init__(self, txt_path, transform=None):
        """
        参数:
            txt_path: TXT 文件路径
            transform: 数据预处理函数
        """
        self.transform = transform
        self.img_info = []  # 存储 (图片路径, 标签)
        
        # 读取 TXT 文件
        with open(txt_path, 'r') as f:
            for line in f:
                # 每行格式: img_path label
                img_path, label = line.strip().split()
                self.img_info.append((img_path, int(label)))
        
        print(f"从 {txt_path} 读取了 {len(self.img_info)} 个样本")
    
    def __getitem__(self, index):
        """获取第 index 个样本"""
        img_path, label = self.img_info[index]
        
        # 读取图片
        img = Image.open(img_path).convert('RGB')
        
        # 应用 transform
        if self.transform is not None:
            img = self.transform(img)
        
        return img, label
    
    def __len__(self):
        return len(self.img_info)

# 示例：创建一个简单的 TXT 文件
txt_content = """data/cat1.jpg 0
data/cat2.jpg 0
data/dog1.jpg 1
data/dog2.jpg 1"""

print("TXT 文件格式示例：")
print(txt_content)

## 3. 形式 2：文件夹结构的 Dataset

In [ ]:
import os
from torch.utils.data import Dataset
from PIL import Image

class FolderDataset(Dataset):
    """
    从文件夹结构读取数据
    目录结构：
    root_dir/
    ├── class1/
    │   ├── img1.jpg
    │   └── img2.jpg
    └── class2/
        ├── img3.jpg
        └── img4.jpg
    """
    
    def __init__(self, root_dir, transform=None):
        """
        参数:
            root_dir: 数据根目录
            transform: 数据预处理函数
        """
        self.root_dir = root_dir
        self.transform = transform
        self.img_info = []  # [(图片路径, 标签), ...]
        
        # 标签映射：文件夹名 -> 数字标签
        self.class_to_idx = {}
        
        self._get_img_info()
    
    def _get_img_info(self):
        """读取数据集信息"""
        # 遍历根目录下的所有文件夹
        for root, dirs, files in os.walk(self.root_dir):
            for file in files:
                if file.endswith(('png', 'jpg', 'jpeg')):
                    # 完整路径
                    img_path = os.path.join(root, file)
                    
                    # 获取类别名（文件夹名）
                    class_name = os.path.basename(root)
                    
                    # 如果是新类别，分配新的标签
                    if class_name not in self.class_to_idx:
                        self.class_to_idx[class_name] = len(self.class_to_idx)
                    
                    label = self.class_to_idx[class_name]
                    self.img_info.append((img_path, label))
        
        print(f"找到 {len(self.img_info)} 个图像")
        print(f"类别映射: {self.class_to_idx}")
    
    def __getitem__(self, index):
        img_path, label = self.img_info[index]
        
        # 读取图片
        img = Image.open(img_path).convert('RGB')
        
        # 应用 transform
        if self.transform is not None:
            img = self.transform(img)
        
        return img, label
    
    def __len__(self):
        return len(self.img_info)

print("文件夹结构 Dataset 示例已定义")
print("推荐：使用 torchvision.datasets.ImageFolder 更方便！")

### 使用 PyTorch 内置的 ImageFolder

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

# ImageFolder 会自动识别文件夹结构
# dataset = ImageFolder(
#     root='data/train',
#     transform=transforms.ToTensor()
# )

# ImageFolder 的优点：
# 1. 自动识别类别
# 2. 自动分配标签
# 3. 代码简洁

print("ImageFolder 使用示例：")
print("dataset = ImageFolder(root='data/train', transform=transform)")
print("dataset.classes       # ['cat', 'dog']")
print("dataset.class_to_idx  # {'cat': 0, 'dog': 1}")

## 4. 形式 3：CSV 文件格式的 Dataset

In [ ]:
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image
import os

class CSVDataset(Dataset):
    """
    从 CSV 文件读取数据
    CSV 格式：
    img_name,label,split
    img1.jpg,0,train
    img2.jpg,1,valid
    """
    
    def __init__(self, root_dir, csv_path, mode='train', transform=None):
        """
        参数:
            root_dir: 图像根目录
            csv_path: CSV 文件路径
            mode: 'train' 或 'valid' 或 'test'
            transform: 数据预处理函数
        """
        self.root_dir = root_dir
        self.transform = transform
        self.mode = mode
        self.img_info = []
        
        self._get_img_info(csv_path)
    
    def _get_img_info(self, csv_path):
        """从 CSV 读取数据信息"""
        # 读取 CSV
        df = pd.read_csv(csv_path)
        
        # 只保留指定模式的数据
        df = df[df['split'] == self.mode]
        df.reset_index(drop=True, inplace=True)
        
        # 遍历表格，获取每个样本信息
        for idx in range(len(df)):
            img_name = df.loc[idx, 'img_name']
            img_path = os.path.join(self.root_dir, img_name)
            label = int(df.loc[idx, 'label'])
            
            self.img_info.append((img_path, label))
        
        print(f"从 CSV 读取了 {len(self.img_info)} 个 {self.mode} 样本")
    
    def __getitem__(self, index):
        img_path, label = self.img_info[index]
        
        # 读取图片
        img = Image.open(img_path).convert('RGB')
        
        # 应用 transform
        if self.transform is not None:
            img = self.transform(img)
        
        return img, label
    
    def __len__(self):
        return len(self.img_info)

# 示例 CSV 内容
csv_example = """img_name,label,split
cat1.jpg,0,train
cat2.jpg,0,train
dog1.jpg,1,train
dog2.jpg,1,valid
cat3.jpg,0,valid"""

print("CSV 文件格式示例：")
print(csv_example)

## 5. 理解 transform 的作用

In [ ]:
from torchvision import transforms
import torch

# transform 是一个函数或函数组合
# 作用：将原始数据转换为模型需要的格式

# 示例 1：单个 transform
transform_single = transforms.ToTensor()

# 示例 2：组合多个 transform
transform_compose = transforms.Compose([
    transforms.Resize((224, 224)),     # 调整大小
    transforms.ToTensor(),              # 转为 Tensor
    transforms.Normalize(               # 标准化
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform 的作用：")
print("1. PIL Image (H, W, C) → Tensor (C, H, W)")
print("2. [0, 255] → [0, 1]")
print("3. 标准化：(x - mean) / std")
print("4. 数据增强：随机翻转、裁剪等")

### 常用的 transforms

In [ ]:
from torchvision import transforms

# 训练集 transform（包含数据增强）
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),           # 调整大小
    transforms.RandomCrop(224),              # 随机裁剪
    transforms.RandomHorizontalFlip(p=0.5),  # 随机水平翻转
    transforms.ColorJitter(                  # 颜色抖动
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 验证集 transform（不做数据增强）
valid_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),    # 中心裁剪
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("训练集和验证集应该使用不同的 transform！")
print("训练集：使用数据增强")
print("验证集：只做基本预处理")

## 6. 完整示例：模拟图像数据集

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

class MockImageDataset(Dataset):
    """模拟图像数据集（用于演示）"""
    
    def __init__(self, num_samples=100, transform=None):
        self.num_samples = num_samples
        self.transform = transform
        
        # 模拟数据信息
        self.img_info = [
            (f"img_{i}.jpg", i % 2)  # 两类：0 和 1
            for i in range(num_samples)
        ]
        
        print(f"创建了 {num_samples} 个模拟样本")
    
    def __getitem__(self, index):
        img_name, label = self.img_info[index]
        
        # 创建随机图像（模拟从硬盘读取）
        img_array = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        
        # 应用 transform
        if self.transform is not None:
            img = self.transform(img)
        
        return img, label
    
    def __len__(self):
        return self.num_samples

# 创建数据集
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

dataset = MockImageDataset(num_samples=10, transform=transform)

# 测试获取数据
img, label = dataset[0]
print(f"\n图像形状: {img.shape}")  # [C, H, W]
print(f"标签: {label}")
print(f"数据类型: {img.dtype}")
print(f"数值范围: [{img.min():.3f}, {img.max():.3f}]")

## 7. Dataset 设计的关键点

In [ ]:
# 关键点 1：初始化时只读取元信息
class GoodDataset(Dataset):
    def __init__(self, data_dir):
        # ✅ 只读取路径列表，不加载图像
        self.img_paths = [...]  # 很快

class BadDataset(Dataset):
    def __init__(self, data_dir):
        # ❌ 加载所有图像到内存
        self.images = [Image.open(p) for p in paths]  # 很慢，占内存

# 关键点 2：__getitem__ 中加载数据
def __getitem__(self, index):
    # 在这里才真正读取图像
    img = Image.open(self.img_paths[index])
    return img

# 关键点 3：transform 在 __getitem__ 中应用
def __getitem__(self, index):
    img = Image.open(self.img_paths[index])
    if self.transform:
        img = self.transform(img)  # 每次获取时才转换
    return img

print("Dataset 设计三原则：")
print("1. 初始化时只读元信息")
print("2. __getitem__ 中才加载数据")
print("3. transform 在 __getitem__ 中应用")

## 8. 练习题

### 练习 1：创建带标签的数字数据集

创建一个数据集，生成随机数字图像（28x28）和对应的标签

In [ ]:
class RandomDigitDataset(Dataset):
    """随机数字数据集"""
    
    def __init__(self, num_samples, transform=None):
        # TODO: 实现初始化
        pass
    
    def __getitem__(self, index):
        # TODO: 生成随机 28x28 灰度图像和标签
        # 提示：使用 np.random.randint 生成图像
        pass
    
    def __len__(self):
        pass

# 测试代码
# dataset = RandomDigitDataset(num_samples=100)
# img, label = dataset[0]
# print(f"图像形状: {img.shape}, 标签: {label}")

### 练习 2：实现一个支持数据增强的 Dataset

为上面的数据集添加 transform 支持

In [ ]:
# 测试不同的 transform
from torchvision import transforms

# transform 1：只转换为 Tensor
transform1 = transforms.ToTensor()

# transform 2：调整大小并转换
transform2 = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# TODO: 创建两个数据集，分别使用不同的 transform
# dataset1 = RandomDigitDataset(100, transform=transform1)
# dataset2 = RandomDigitDataset(100, transform=transform2)

# TODO: 比较输出的图像形状

## 9. 小结

### 数据集组织形式对比

| 形式 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| TXT | 简单直观 | 需要手动维护 | 小型数据集 |
| 文件夹 | 直观易管理 | 不适合多标签 | 单标签分类 |
| CSV | 灵活，支持元信息 | 需要 pandas | 复杂数据集 |

### 核心要点

1. **延迟加载**：初始化时不加载数据
2. **transform**：在 `__getitem__` 中应用
3. **类别映射**：字符串标签 → 数字标签
4. **训练/验证**：使用不同的 transform

### 最佳实践

```python
# 推荐的 Dataset 模板
class MyDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        # 1. 只读取路径列表
        self.img_paths = [...]
        self.labels = [...]
        self.transform = transform
    
    def __getitem__(self, index):
        # 2. 读取单个样本
        img = Image.open(self.img_paths[index])
        label = self.labels[index]
        
        # 3. 应用 transform
        if self.transform:
            img = self.transform(img)
        
        return img, label
    
    def __len__(self):
        return len(self.img_paths)
```

### 下一步

在下一个 notebook 中，我们将学习如何使用 **DataLoader** 批量加载数据！

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！